# SageMaker Studio Demo: MLflow Tracking

- Dataset: [Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) (UCI)
- Same two models as `studio_train.ipynb`, logged to MLflow instead of a
  pandas table that dies with the kernel

Steps

1. setup notebook, install Python libraries
2. load `featured/hour.parquet` from S3
3. train model, log to MLflow
   - baseline
   - deeper tree
4. query the runs back and compare
5. visualize
6. register the model

---

## 1. Setup

`sagemaker-mlflow` is the plugin that signs MLflow client calls with
SigV4. The Studio image may not ship it.

In [ ]:
%pip install --quiet mlflow sagemaker-mlflow

The tracking URI is the app ARN, not an https URL:

```
terraform -chdir=infra output -raw data_bucket
terraform -chdir=infra output -raw mlflow_app_arn
```

In [ ]:
import io

import boto3
import mlflow
import numpy as np
import pandas as pd

REGION = "ca-central-1"
BUCKET = "mlops-sagemaker-studio-dev-data-3vi8kw"
TRACKING_ARN = "arn:aws:sagemaker:ca-central-1:099139718958:mlflow-app/app-7BMBBX3TS2CZ"

FEATURED_KEY = "featured/hour.parquet"
EXPERIMENT = "bike-sharing"

s3 = boto3.client("s3", region_name=REGION)

mlflow.set_tracking_uri(TRACKING_ARN)
mlflow.set_experiment(EXPERIMENT)

print(f"tracking uri: {mlflow.get_tracking_uri()}")

---

## 2. Load `featured/hour.parquet`

The features written by `studio_train.ipynb`, with the same 2011/2012
split so the runs stay comparable.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=FEATURED_KEY)
df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

TARGET = "cnt"
FEATURES = [c for c in df.columns if c != TARGET]

train = df[df.yr == 0]
test = df[df.yr == 1]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {len(train)} rows (2011)   test {len(test)} rows (2012)")
print(f"{len(FEATURES)} features: {FEATURES}")

---

## 3. Train and log to MLflow

Same two random forests as `studio_train.ipynb`:

- **baseline** -- `min_samples_leaf=5`
- **deeper** -- `min_samples_leaf=1`, fully grown trees

Each run logs its params, metrics and the model itself.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


RUNS = {
    "baseline": {"n_estimators": 100, "min_samples_leaf": 5},
    "deeper": {"n_estimators": 100, "min_samples_leaf": 1},
}

Re-running this notebook would otherwise pile up a new copy of each run
every time, so clear the previous ones first. The SageMaker MLflow
backend rejects `or` in a filter string, hence one query per name.

In [ ]:
client = mlflow.MlflowClient()
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT).experiment_id

for name in RUNS:
    for stale in client.search_runs(
        [experiment_id], filter_string=f"tags.mlflow.runName = '{name}'"
    ):
        client.delete_run(stale.info.run_id)
        print(f"deleted stale run {name} {stale.info.run_id[:8]}")

In [ ]:
models = {}
results = {}
model_uris = {}

for name, params in RUNS.items():
    with mlflow.start_run(run_name=name):
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(X_train, y_train)

        metrics = evaluate(y_test, model.predict(X_test))

        buf = io.BytesIO()
        joblib.dump(model, buf)
        metrics["size_mb"] = buf.tell() / 1024 / 1024

        mlflow.log_params(params)
        mlflow.log_param("split", "2011-train/2012-test")
        mlflow.log_metrics(metrics)

        # Ships the model to the artifact store, not just the numbers.
        info = mlflow.sklearn.log_model(model, name="model")
        model_uris[name] = info.model_uri

        models[name], results[name] = model, metrics
        print(f"{name:9} rmse={metrics['rmse']:6.1f}  mae={metrics['mae']:5.1f}  "
              f"r2={metrics['r2']:.3f}  {metrics['size_mb']:.1f}MB")

---

## 4. Query the runs back

This is what a pandas table cannot do -- the runs outlive the kernel and
are readable by anyone with access to the app.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT], order_by=["metrics.rmse ASC"]
)

cols = [
    "tags.mlflow.runName",
    "params.min_samples_leaf",
    "metrics.rmse",
    "metrics.mae",
    "metrics.r2",
    "metrics.size_mb",
]

runs[cols].round(2)

The deeper tree is 0.9% better on rmse and 6x the size.

---

## 5. Visualize

In [ ]:
import matplotlib.pyplot as plt

BEST = "baseline"
model = models[BEST]
pred = model.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle(f"Bike sharing demand -- 2012 holdout ({BEST})")

# rmse vs size
ax = axes[0]
for name, m in results.items():
    ax.scatter(m["size_mb"], m["rmse"], s=90)
    ax.annotate(name, (m["size_mb"], m["rmse"]),
                textcoords="offset points", xytext=(8, 4))
ax.set_xlabel("artifact size (MB)")
ax.set_ylabel("rmse")
ax.set_title("accuracy vs size")

# feature importance
ax = axes[1]
imp = pd.Series(model.feature_importances_, index=FEATURES).nlargest(8)
ax.barh(imp.index[::-1], imp.values[::-1])
ax.set_title("feature importance")

# average day
ax = axes[2]
by_hour = pd.DataFrame({"hr": X_test.hr, "actual": y_test, "predicted": pred})
by_hour = by_hour.groupby("hr").mean()
ax.plot(by_hour.index, by_hour.actual, label="actual", lw=2)
ax.plot(by_hour.index, by_hour.predicted, label="predicted", lw=2, ls="--")
ax.set_xlabel("hour of day")
ax.set_ylabel("mean rides")
ax.set_title("average day")
ax.set_xticks(range(0, 24, 3))
ax.legend()

fig.tight_layout()
plt.show()

The MLflow UI plots the same comparison across runs -- select both, then
Compare:

```
terraform -chdir=infra output -raw mlflow_ui_command
```

---

## 6. Register the model

The baseline is 0.9% worse on rmse and 6x smaller. A serverless endpoint
reloads the model on every cold start, so the size wins.

In [ ]:
result = mlflow.register_model(
    model_uri=model_uris[BEST], name="bike-sharing-rf"
)

print(f"registered {result.name} version {result.version}")